In [1]:
import pandas as pd

In [3]:
tf_link_path = "/projects/b1042/GoyalLab/jaekj/TWINFER/TF_Link/tf_link_data/all_tf_interactions_experimental_only.csv"
infer_path = "/projects/b1042/GoyalLab/jaekj/TWINFER/additional_data/Twinfer_TF_gene/directional_gene_correlation_inference_LARRY.csv"

df_tf = pd.read_csv(tf_link_path)
df_inf = pd.read_csv(infer_path)

In [4]:
# Transcription factor & target
# Normalize gene symbols

# From the TF dataset
df_tf["Name.TF"] = df_tf["Name.TF"].str.strip()
df_tf["Name.Target"] = df_tf["Name.Target"].str.strip()

# From the inferred gene
df_inf["gene_1"] = df_inf["gene_1"].str.strip()
df_inf["gene_2"] = df_inf["gene_2"].str.strip()

In [5]:
# TF genes: TF -> TF
tf_genes = set(df_tf["Name.TF"])

df_tf_tf = df_tf[
    (df_tf["Name.TF"].isin(tf_genes)) &
    (df_tf["Name.Target"].isin(tf_genes))
]

In [6]:
tf_link_edges = set(
    zip(df_tf_tf["Name.TF"], df_tf_tf["Name.Target"])
)

In [7]:
# Score the inferred edges 
df_inf["supported_by_TFLink"] = df_inf.apply(
    lambda r: (r["gene_1"], r["gene_2"]) in tf_link_edges,
    axis=1
)

df_inf

,gene_1,gene_2,type,supported_by_TFLink
0,Gata2,Klf1,direction_inferred,False
1,Gata2,Gata1,direction_inferred_maybe,True
2,Cebpa,Spi1,direction_inferred,True
3,Tal1,Gata2,direction_inferred_maybe,True
4,Spi1,Jun,direction_inferred,True
5,Egr1,Gata2,direction_inferred,False
6,Klf1,Gata2,direction_inferred,False
7,Gata1,Gata2,direction_inferred_maybe,False
8,Gata2,Egr1,direction_inferred,False
9,Gata2,Tal1,direction_inferred_maybe,True


In [8]:
df_inf["supported_by_TFLink"].value_counts()

supported_by_TFLink
False    5
True     5
Name: count, dtype: int64

In [9]:
df_inf["reverse_supported"] = df_inf.apply(
    lambda r: (r["gene_2"], r["gene_1"]) in tf_link_edges,
    axis=1
)

In [8]:
def classify(row):
    if row.supported_by_TFLink:
        return "directionally_supported"
    if row.reverse_supported:
        return "direction_conflict"
    return "unsupported"

df_inf["TFLink_status"] = df_inf.apply(classify, axis=1)
df_inf["TFLink_status"].value_counts()

In [9]:
import numpy as np

rng = np.random.default_rng(0)
tf_list = sorted(tf_genes)

n_edges = len(df_inf)
n_perm = 1000

null_support = []

for _ in range(n_perm):
    rand_edges = set()
    while len(rand_edges) < n_edges:
        g1, g2 = rng.choice(tf_list, size=2, replace=False)
        rand_edges.add((g1, g2))
    
    count = sum(e in tf_link_edges for e in rand_edges)
    null_support.append(count)

In [ ]:
obs = df_inf["supported_by_TFLink"].sum()

In [ ]:
pval = (np.sum(np.array(null_support) >= obs) + 1) / (n_perm + 1)
obs, pval

In [ ]:
import matplotlib.pyplot as plt

plt.hist(null_support, bins=30, alpha=0.7)
plt.axvline(obs, color="red", linewidth=2, label="Observed")
plt.xlabel("TF-Link supported edges")
plt.ylabel("Count")
plt.legend()
plt.title("TF-Link enrichment of inferred TF→TF edges")
plt.show()